## 3.15 Random Values: SFI Backend QA Test Garage

I am a QA engineer stress-testing an SFI automotive safety-spec backend. Randomness picks which part record gets tested and which API action hits it, so many runs exercise many different paths through the backend.

Two rules carry the whole lesson:

- `random.randint(a, b)` includes **both** endpoints, just like College Board `RANDOM(a, b)`.
- Repeats are normal. One run only shows one of the possible outcomes, so correctness means reasoning about **every** possible output, not the one I happened to see.

**About the outputs below:** the Popcorn Hack cells are not seeded, so the random values change every run and yours will differ. Every cell also prints a check line (like `All IDs valid: True`) that is the same on every run. The homework has one seeded run so its output is exact and repeatable for debugging.


## Popcorn Hack 1: Random SFI Record Number

**Prediction.** `random.randint(1, record_count)` with `record_count = 4` can return **1, 2, 3, or 4**. Both endpoints are included, so record 4 is reachable and 0 or 5 never appears. I cannot predict the exact first value, only that it is one of those four. Over 10 runs I expect to see repeats, because each call is independent and does not remember earlier picks.


In [ ]:
# CODE_RUNNER: 3.15 Popcorn 1 - Random SFI Record Number

import random

record_count = 4
runs = 10

record_ids = []
for run in range(runs):
    record_ids.append(random.randint(1, record_count))

print("Possible record IDs:", list(range(1, record_count + 1)))
print("Selected record IDs:", record_ids)
print("All IDs valid:", all(1 <= rid <= record_count for rid in record_ids))
print("Repeats seen:", len(record_ids) != len(set(record_ids)))


**Changing one input.** I changed `record_count` from 4 to 6. Now the possible outputs are 1 through 6, and the same check still holds because the range is tied to `record_count` instead of a hard-coded 4.


In [ ]:
# CODE_RUNNER: 3.15 Popcorn 1 - record_count changed to 6

import random

record_count = 6
runs = 10

record_ids = []
for run in range(runs):
    record_ids.append(random.randint(1, record_count))

print("Possible record IDs:", list(range(1, record_count + 1)))
print("Selected record IDs:", record_ids)
print("All IDs valid:", all(1 <= rid <= record_count for rid in record_ids))
print("Repeats seen:", len(record_ids) != len(set(record_ids)))


**Actual output.** The `Selected record IDs` list is different on every run (your numbers will differ), but every value is always between 1 and `record_count`, so `All IDs valid: True` prints every time. The first two lines are always:

    Possible record IDs: [1, 2, 3, 4]
    Possible record IDs: [1, 2, 3, 4, 5, 6]

for the two cells. `Repeats seen` is always `True`: with only 4 IDs and 10 picks, a repeat is guaranteed (10 picks cannot all be different when there are only 4 values). With 6 IDs it is still guaranteed, since 10 > 6.

**Why repeats are allowed.** Random selection is not rotation. Each `randint` call is independent, so record 2 can come up twice in a row. Seeing 1, 3, 3 does not mean record 4 is unreachable. It just did not come up yet.


## Popcorn Hack 2: Random Structured SFI Car Part

**Prediction.** `random.choice(parts)` returns one whole dictionary, so the product name, category, and spec number printed always belong together. There are 3 possible results. For example, if it picks spec 1.2 it must say "Multiple Disc Clutch Assemblies" and "Drag Racing", never a mix of fields from two different records.

The wrong way would be to pick each field separately (a random name and a random spec), which could pair "Racing Flywheel Record" with spec 1.1 and break the record.


In [ ]:
# CODE_RUNNER: 3.15 Popcorn 2 - Random SFI Part Record

import random

parts = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

selected = random.choice(parts)

print("Product name:", selected["product_name"])
print("Category:", selected["category"])
print("Spec number:", selected["spec_number"])
print("Record intact:", selected in parts)


**Changing one input: choosing by index instead.** College Board lists start at 1, but Python indexes start at 0. To pick by index, the valid range is `0` to `len(parts) - 1`. The opposite case is the off-by-one range `randint(0, len(parts))`, which can produce index 3 and crash. This cell picks a valid index, then deliberately tries index 3 to show what the bad range would hit.


In [ ]:
# CODE_RUNNER: 3.15 Popcorn 2 - choose by index (valid vs invalid range)

import random

parts = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

index = random.randint(0, len(parts) - 1)
part = parts[index]
print("Valid index range: 0 to", len(parts) - 1)
print("Index", index, "->", part["spec_number"], part["product_name"], "(" + part["category"] + ")")

bad_index = len(parts)
try:
    print(parts[bad_index])
except IndexError:
    print("randint(0, len(parts)) could give index", bad_index, "-> IndexError, so that range is invalid")


**Actual output.** The selected record changes between runs (yours will differ), but the three fields always match one real record and `Record intact: True` prints every time. In the index version, the index is always 0, 1, or 2, and the last line is always:

    randint(0, len(parts)) could give index 3 -> IndexError, so that range is invalid

**Takeaway.** `random.choice` is the safest way to pick a structured record: it can only return a real item, so there is no index range to get wrong.


## Popcorn Hack 3: Random SFI API Test

**Prediction of two possible outcomes.** There are 3 parts × 3 actions = **9** possible part/action pairs.

- If it picks spec 1.2 with `GET/search`, it should print `200 OK: found Multiple Disc Clutch Assemblies (spec 1.2)`.
- If it picks spec 2.1 with `POST/create`, it should print a validation message for creating `Racing Flywheel Record` with spec 2.1.

The action decides which `if` branch runs, so a different random action means a different backend path. Each action gets its own branch, so none of the three is left without a response.


In [ ]:
# CODE_RUNNER: 3.15 Popcorn 3 - Random SFI API Test

import random

parts = [
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"}
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update"
]


def run_api_test(part, action):
    name = part["product_name"]
    spec = part["spec_number"]
    if action == "GET/search":
        return "200 OK: found " + name + " (spec " + spec + ")"
    elif action == "POST/create":
        return "201 Created: validated and created " + name + " with spec " + spec
    elif action == "PUT/update":
        return "200 OK: updated " + name + " (spec " + spec + ")"
    else:
        return "400 Bad Request: unknown action " + action


for test in range(1, 4):
    part = random.choice(parts)
    action = random.choice(actions)
    print("Test", test, "|", action, "| spec", part["spec_number"])
    print("   ", run_api_test(part, action))

print("Possible part/action pairs:", len(parts) * len(actions))


**Changing the action.** To make sure every branch really works (not just the ones that happened to come up randomly), this cell fixes the part to spec 1.2 and runs each action once, plus an action that is not in the list.


In [ ]:
# CODE_RUNNER: 3.15 Popcorn 3 - every action on spec 1.2

import random

parts = [
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"}
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update"
]


def run_api_test(part, action):
    name = part["product_name"]
    spec = part["spec_number"]
    if action == "GET/search":
        return "200 OK: found " + name + " (spec " + spec + ")"
    elif action == "POST/create":
        return "201 Created: validated and created " + name + " with spec " + spec
    elif action == "PUT/update":
        return "200 OK: updated " + name + " (spec " + spec + ")"
    else:
        return "400 Bad Request: unknown action " + action


part = parts[1]
for action in actions + ["PATCH/unknown"]:
    print(action, "->", run_api_test(part, action))


**Actual output.** The random cell prints 3 tests whose part and action change every run (yours will differ), always ending with `Possible part/action pairs: 9`. The fixed cell is the same every run:

    GET/search -> 200 OK: found Multiple Disc Clutch Assemblies (spec 1.2)
    POST/create -> 201 Created: validated and created Multiple Disc Clutch Assemblies with spec 1.2
    PUT/update -> 200 OK: updated Multiple Disc Clutch Assemblies (spec 1.2)
    PATCH/unknown -> 400 Bad Request: unknown action PATCH/unknown

That matches my prediction for spec 1.2 + `GET/search`. Changing the action changed the result because the action picks which branch runs.


## MCQ Check

**MCQ 3.15: 4/4 | answers: C,B,B,B**

| # | Question | Answer | Why |
|---|---|---|---|
| 1 | What values can `random.randint(2, 5)` return? | **C**: 2, 3, 4, 5 | `randint` includes both endpoints, and it only returns integers. |
| 2 | Record A appears twice in a row. What does that mean? | **B**: Repeats are allowed in random selection | Each choice is independent, so the same record can come up again. It does not mean the code is broken. |
| 3 | Which Python expression matches `RANDOM(1,4)`? | **B**: `random.randint(1,4)` | Both return an integer from 1 through 4 inclusive. `randint(0,4)` adds 0, and `choice([1,2,3])` is missing 4. |
| 4 | Why is testing a random program only once insufficient? | **B**: One run may show only one of many possible outcomes | Correctness depends on every possible output, not the one I saw. |


## Pseudocode Check

Tracing the Section 3 College Board pseudocode by hand with `LENGTH(parts) = 3`, choosing `recordIndex = 2` and `actionNumber = 3`:

- `selectedPart ← parts[2]` is the **second** record (College Board lists start at 1), which is spec 1.2, Multiple Disc Clutch Assemblies. In Python that same record is `parts[1]`.
- `actionNumber = 3`: not 1, not 2, so it reaches the inner `IF (actionNumber = 3)` and displays **PUT/update**.

`recordIndex` can be 1, 2, or 3 and `actionNumber` can be 1 to 4, so there are 3 × 4 = **12** possible pairs. My homework simulator uses the same four actions with `random.choice`, which matches the lesson's Python equivalent.


## Homework Hack: SFI Backend QA Simulator

**Plan.** Each test randomly picks one complete part record and one of four actions, then routes it through a backend-style handler:

| Action | Spec already exists | Spec does not exist |
|---|---|---|
| `GET/search` | 200 OK, found | 404 Not Found |
| `POST/create` | **409 Conflict, duplicate spec rejected** | 201 Created, spec added |
| `PUT/update` | 200 OK, updated | 404 Not Found |
| `DELETE/remove` | 200 OK, removed | 404 Not Found |

**Duplicate-spec handling.** `POST/create` checks `existing_spec_numbers` first and refuses to add a spec that is already there. After every test, the simulator checks that no spec number appears twice. That is the "invalid state" the lesson's QA question asks about.

**Reusable.** `run_qa_simulation(parts, actions, existing_spec_numbers, test_count, seed)` works with any list of parts, any number of tests, and copies the starting spec list so each run starts fresh. `seed=None` gives normal changing runs; a fixed seed makes one run repeatable for debugging.

**Possible outputs.** With 3 parts and 4 actions, each test has 3 × 4 = 12 possible part/action pairs, and any pair can repeat.

**Prediction for the seeded run (seed 3).** I cannot know the exact picks ahead of time, but I can predict the rules: a `POST/create` on spec 1.1 or 2.1 (already in the backend) must be rejected with 409, and the final spec list can never contain a duplicate.


In [ ]:
# CODE_RUNNER: 3.15 Homework - SFI Backend QA Simulator

import random

parts = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update",
    "DELETE/remove"
]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5


def handle_request(part, action, specs):
    name = part["product_name"]
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in specs:
            return "200 OK: found " + name + " (spec " + spec + ")"
        return "404 Not Found: spec " + spec + " is not in the backend"
    elif action == "POST/create":
        if spec in specs:
            return "409 Conflict: spec " + spec + " already exists, create rejected"
        specs.append(spec)
        return "201 Created: added " + name + " (spec " + spec + ")"
    elif action == "PUT/update":
        if spec in specs:
            return "200 OK: updated " + name + " (spec " + spec + ")"
        return "404 Not Found: cannot update missing spec " + spec
    elif action == "DELETE/remove":
        if spec in specs:
            specs.remove(spec)
            return "200 OK: removed " + name + " (spec " + spec + ")"
        return "404 Not Found: cannot remove missing spec " + spec
    return "400 Bad Request: unknown action " + action


def run_qa_simulation(parts, actions, existing_spec_numbers, test_count, seed=None):
    rng = random.Random(seed)
    specs = list(existing_spec_numbers)
    action_counts = {}
    for action in actions:
        action_counts[action] = 0

    print("QA run | tests:", test_count, "| seed:", seed, "| starting specs:", specs)
    for test in range(1, test_count + 1):
        part = rng.choice(parts)
        action = rng.choice(actions)
        action_counts[action] = action_counts[action] + 1
        result = handle_request(part, action, specs)
        print("Test", test, "|", action, "|", part["spec_number"], part["product_name"], "(" + part["category"] + ")")
        print("   ", result)
        if len(specs) != len(set(specs)):
            print("    INVALID STATE: duplicate spec numbers", specs)

    print("Final specs:", specs)
    print("Action counts:", action_counts)
    print("No duplicate specs:", len(specs) == len(set(specs)))
    print()
    return specs


# Normal run: different every time
run_qa_simulation(parts, actions, existing_spec_numbers, test_count)

# Repeatable run for debugging: same output every time
run_qa_simulation(parts, actions, existing_spec_numbers, test_count, seed=3)


**Actual output, normal run.** The first block changes every run (yours will differ). It always shows 5 tests, each line has one of the 3 parts and one of the 4 actions, the action counts add up to 5, and it ends with `No duplicate specs: True`.

**Actual output, seeded run (seed 3).** This block is identical on every run:

    QA run | tests: 5 | seed: 3 | starting specs: ['1.1', '2.1']
    Test 1 | POST/create | 1.1 Replacement Flywheels (Auto Racing)
        409 Conflict: spec 1.1 already exists, create rejected
    Test 2 | DELETE/remove | 1.2 Multiple Disc Clutch Assemblies (Drag Racing)
        404 Not Found: cannot remove missing spec 1.2
    Test 3 | GET/search | 2.1 Racing Flywheel Record (Auto Racing)
        200 OK: found Racing Flywheel Record (spec 2.1)
    Test 4 | GET/search | 2.1 Racing Flywheel Record (Auto Racing)
        200 OK: found Racing Flywheel Record (spec 2.1)
    Test 5 | PUT/update | 1.2 Multiple Disc Clutch Assemblies (Drag Racing)
        404 Not Found: cannot update missing spec 1.2
    Final specs: ['1.1', '2.1']
    Action counts: {'GET/search': 2, 'POST/create': 1, 'PUT/update': 1, 'DELETE/remove': 1}
    No duplicate specs: True

That run hit all four actions, and the `POST/create` on an existing spec was rejected with 409 instead of adding a duplicate, as predicted. Tests 3 and 4 picked the exact same pair (GET/search on spec 2.1), which shows repeats are normal. The final spec list has no duplicates.


## Homework Tests: directed cases

Random runs might not hit every rule, so this cell calls the same `handle_request` on chosen cases to prove each rule works. It starts fresh from `["1.1", "2.1"]`.


In [ ]:
# CODE_RUNNER: 3.15 Homework Tests - directed duplicate and missing cases

import random

parts = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update",
    "DELETE/remove"
]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5


def handle_request(part, action, specs):
    name = part["product_name"]
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in specs:
            return "200 OK: found " + name + " (spec " + spec + ")"
        return "404 Not Found: spec " + spec + " is not in the backend"
    elif action == "POST/create":
        if spec in specs:
            return "409 Conflict: spec " + spec + " already exists, create rejected"
        specs.append(spec)
        return "201 Created: added " + name + " (spec " + spec + ")"
    elif action == "PUT/update":
        if spec in specs:
            return "200 OK: updated " + name + " (spec " + spec + ")"
        return "404 Not Found: cannot update missing spec " + spec
    elif action == "DELETE/remove":
        if spec in specs:
            specs.remove(spec)
            return "200 OK: removed " + name + " (spec " + spec + ")"
        return "404 Not Found: cannot remove missing spec " + spec
    return "400 Bad Request: unknown action " + action


specs = list(existing_spec_numbers)
cases = [
    (parts[0], "POST/create"),
    (parts[1], "POST/create"),
    (parts[1], "POST/create"),
    (parts[2], "DELETE/remove"),
    (parts[2], "GET/search"),
    (parts[2], "PUT/update"),
    (parts[0], "PUT/update"),
]
for part, action in cases:
    print(action, part["spec_number"], "->", handle_request(part, action, specs))
print("Final specs:", specs)
print("No duplicate specs:", len(specs) == len(set(specs)))


**Actual output** (same every run, no randomness):

    POST/create 1.1 -> 409 Conflict: spec 1.1 already exists, create rejected
    POST/create 1.2 -> 201 Created: added Multiple Disc Clutch Assemblies (spec 1.2)
    POST/create 1.2 -> 409 Conflict: spec 1.2 already exists, create rejected
    DELETE/remove 2.1 -> 200 OK: removed Racing Flywheel Record (spec 2.1)
    GET/search 2.1 -> 404 Not Found: spec 2.1 is not in the backend
    PUT/update 2.1 -> 404 Not Found: cannot update missing spec 2.1
    PUT/update 1.1 -> 200 OK: updated Replacement Flywheels (spec 1.1)
    Final specs: ['1.1', '1.2']
    No duplicate specs: True

Every rule is covered: a duplicate on a starting spec (1.1) is rejected, a new spec (1.2) is created once and then rejected the second time, and search/update on a removed spec (2.1) return 404 instead of acting on a record that is not there.


## Submission Notes

    Lesson: CSP 3.15 Random Values
    Notebook: https://rex0dex.github.io/portfolio/python/random-hw
    MCQ 3.15: 4/4 | answers: C,B,B,B
    Popcorn Hacks 1-3 completed during class and copied to notebook: yes
    Homework tested: 5-test unseeded runs (always valid, no duplicate specs) + seed 3 run hitting all 4 actions with a 409 duplicate rejection + 7 directed cases (409 on 1.1, 201 then 409 on 1.2, DELETE 2.1 then GET/PUT 2.1 -> 404)
